In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sys
import os
from sklearn import set_config

from sksurv.linear_model import CoxnetSurvivalAnalysis
from sklearn.model_selection import train_test_split
set_config(display="text")  # displays text representation of estimators

from sksurv.util import Surv
sys.path.append(os.path.abspath("../../"))
from src.utils.ConvertTextToCsv import TextToCsv
from src.utils.Preprocessing import Preprocessor
from src.utils.cox_models import Cox_regression, p_values_Cox_regression, plot_coefficients
%load_ext autoreload
%autoreload 2
%matplotlib inline


In [ ]:
pp = Preprocessor()
cox_lasso = CoxnetSurvivalAnalysis(l1_ratio=1.0, alpha_min_ratio=0.01)
df_clinical_data = pd.read_csv("../../data/raw/brca_tcga_pub2015_clinical_data.tsv", sep='\t')
df_clinical_keep = pp.clean_columns_dataset(df_clinical_data)


<font size="4">Preprocessing</font>

In [ ]:
list_df = pp.total_type_len_type_cancer(df_clinical_keep)
df_clinical_keep["Tumor-Cancer"] = list_df
df_mRNA_transformed = TextToCsv("../../data/raw/data_mrna_seq_v2_rsem.txt")
clean_df = pp.elimnation_zeros(df_mRNA_transformed, "Hugo_Symbol")

<font size="4">ESR1 coefficients and p_values</font>

In [ ]:
df_ESR1 = pp.gene_to_long(clean_df,"ESR1")

In [ ]:
df_ESR1_merged = df_ESR1.merge(df_clinical_keep, on="Sample ID", how="inner")
status = df_ESR1_merged["Overall Survival Status"].astype(str).str.strip()
df_ESR1_merged["event"] = status.str.contains("DECEASED", na=False) 
df_ESR1_merged = df_ESR1_merged.dropna(subset=["Overall Survival (Months)"])
X_ESR1 = df_ESR1_merged[["expression"]]
Y_surv_ESR1 = Surv.from_dataframe(
    event="event",
    time="Overall Survival (Months)",
    data=df_ESR1_merged
)

In [ ]:
X_train_ESR1, X_test_ESR1, Y_train_ESR1, Y_test_ESR1 = train_test_split(
    X_ESR1, Y_surv_ESR1, train_size=0.80, test_size=0.20, random_state=42
)


In [ ]:
betas_ESR1, chp_predict_ESR1, survival_curve_ESR1, risk_curve_ESR1 = Cox_regression(X_train_ESR1, Y_train_ESR1, X_test_ESR1)

In [ ]:
print(betas_ESR1)
#Alpha is regularization 
#Low alpha == Low regularization
#Big alpha == Big regularization

#Espression == Betas 
#β > 0 → aumenta el riesgo
#β < 0 → reduce el riesgo
#β ≈ 0 → no hay efecto

In [ ]:
print(chp_predict_ESR1)

# Risk Score = Expression x β
# valor más alto → mayor riesgo relativo
#valor más bajo → menor riesgo relativo

In [ ]:
for fn in survival_curve_ESR1:
    plt.step(fn.x, fn(fn.x), where="post")

plt.title("Survival curve for ESR1")
plt.xlabel("Days")
plt.ylim(0, 1)
plt.ylabel("% of survival")
plt.show()

In [ ]:
for fn in risk_curve_ESR1:
    plt.step(fn.x, fn(fn.x), where="post")

plt.title("Risk curve for ESR1")
plt.xlabel("Days")
plt.ylim(0, 1)
plt.ylabel("% of risk")
plt.show()

In [ ]:
plot_coefficients(betas_ESR1, n_highlight=5, title="Coefficients of ESR1")

In [ ]:
"""cox_lasso.fit(X_train_ESR1, X_test_ESR1)
coefficient_lasso = pd.DataFrame(cox_lasso.coef_, index=X.columns, columns=(cox_lasso.alphas_))
plot_coefficients(coefficient_lasso, n_highlight=5, title="ESR1 Coefficients")
cox_lasso.predict(X_test)
"""

In [ ]:
df_life_line_ESR1 = df_ESR1_merged[["expression", "event", "Overall Survival (Months)"]]

In [ ]:
p_values_Cox_regression(df_life_line_ESR1,event_col="event", duration_col="Overall Survival (Months)")

#p_values = p
#p < 0.05 → hay evidencia de que la variable sí afecta la supervivencia
#p ≥ 0.05 → no hay evidencia suficiente
# p == 0.79 = No significativo

<font size="4">AURKA coefficients and p_values</font>

In [ ]:
df_AURKA = pp.gene_to_long(clean_df, "AURKA")

In [ ]:
df_AURKA_merged = df_AURKA.merge(df_clinical_keep, on="Sample ID", how="inner")
status = df_AURKA_merged["Overall Survival Status"].astype(str).str.strip()
df_AURKA_merged["event"] = status.str.contains("DECEASED", na=False) 
df_AURKA_merged = df_AURKA_merged.dropna(subset=["Overall Survival (Months)"])
X_AURKA = df_AURKA_merged[["expression"]]
Y_surv_AURKA = Surv.from_dataframe(
    event="event",
    time="Overall Survival (Months)",
    data=df_AURKA_merged
)

In [ ]:
X_train_AURKA, X_test_AURKA, Y_train_AURKA, Y_test_AURKA = train_test_split(
    X_AURKA, Y_surv_AURKA, train_size=0.80, test_size=0.20, random_state=42
)


In [ ]:
betas_AURKA, chp_predict_AURKA, survival_curve_AURKA, risk_curve_AURKA = Cox_regression(X_train_AURKA, Y_train_AURKA, X_test_AURKA)

In [ ]:
print(betas_AURKA)
#Alpha is regularization 
#Low alpha == Low regularization
#Big alpha == Big regularization

#Espression == Betas 
#β > 0 → aumenta el riesgo
#β < 0 → reduce el riesgo
#β ≈ 0 → no hay efecto

In [ ]:
print(chp_predict_AURKA)

# Risk Score = Expression x β
# valor más alto → mayor riesgo relativo
#valor más bajo → menor riesgo relativo

In [ ]:
for fn in survival_curve_AURKA:
    plt.step(fn.x, fn(fn.x), where="post")

plt.title("Survival curve for AURKA")
plt.xlabel("Days")
plt.ylim(0, 1)
plt.ylabel("% of survival")
plt.show()

In [ ]:
for fn in risk_curve_AURKA:
    plt.step(fn.x, fn(fn.x), where="post")

plt.title("Risk curve for AURKA")
plt.ylim(0, 1)
plt.xlabel("Days")
plt.ylabel("% of survival")
plt.show()

In [ ]:
plot_coefficients(betas_AURKA, n_highlight=5, title="Coefficients of ESR1")

In [ ]:
df_life_line_AURKA = df_AURKA_merged[["expression", "event", "Overall Survival (Months)"]]

In [ ]:
p_values_Cox_regression(df_life_line_AURKA,event_col="event", duration_col="Overall Survival (Months)")

#p_values = p
#p < 0.05 → hay evidencia de que la variable sí afecta la supervivencia
#p ≥ 0.05 → no hay evidencia suficiente
# p == 0.61 = No significativo

<font size="4">Luminal A betas and p_values</font>

In [ ]:
pp = Preprocessor()
df_clinical_data = pd.read_csv("../../data/raw/brca_tcga_pub2015_clinical_data.tsv", sep='\t')
df_clinical_keep = pp.clean_columns_dataset(df_clinical_data)
list_df = pp.total_type_len_type_cancer(df_clinical_keep)
df_clinical_keep["Tumor-Cancer"] = list_df
df_mRNA_transformed = TextToCsv("../../data/raw/data_mrna_seq_v2_rsem.txt")
df_merged = pp.merge_datasets(df_clinical_keep, df_mRNA_transformed)

In [ ]:
expressions_genes_cols = df_merged.iloc[1:20441].sample(650, axis="columns")
cols = ["Tumor-Cancer", "Overall Survival Status", "Overall Survival (Months)"] + list(expressions_genes_cols)

comparation_df = df_merged.loc[
    df_merged["Tumor-Cancer"].isin(["Luminal A", "Luminal B", "TNBC", "HER2-enriched"]),
    cols
]

comparation_df

In [ ]:
comparation_df = pp.elimnation_zeros(comparation_df, "Tumor-Cancer")

In [ ]:
luminal_A_df = comparation_df[comparation_df["Tumor-Cancer"] == "Luminal A"]
status = luminal_A_df["Overall Survival Status"].astype(str).str.strip()
luminal_A_df["event"] = status.str.contains("DECEASED", na=False) 
luminal_A_df = luminal_A_df.dropna(subset=["Overall Survival (Months)"])
X_LUMINAL_A = luminal_A_df.iloc[:, 3:-1]
Y_surv_LUMINAL_A = Surv.from_dataframe(
    event="event",
    time="Overall Survival (Months)",
    data=luminal_A_df
)

In [ ]:

X_train_LuminalA, X_test_LuminalA,Y_train_LuminalA,Y_test_LuminalA = train_test_split(
    X_LUMINAL_A, Y_surv_LUMINAL_A, train_size=0.80, test_size=0.20, random_state=42
)


In [ ]:
betas_LuminalA, chp_predict_LuminalA, survival_curve_LuminalA, risk_curve_LuminalA = Cox_regression(X_train_LuminalA, Y_train_LuminalA, X_test_LuminalA)

In [ ]:
for fn in survival_curve_LuminalA:
    plt.step(fn.x, fn(fn.x), where="post")

plt.title("Survival curve for ESR1")
plt.xlabel("Days")
plt.ylim(0, 1)
plt.ylabel("% of survival")
plt.show()

In [ ]:
for fn in risk_curve_LuminalA:
    plt.step(fn.x, fn(fn.x), where="post")

plt.title("Survival curve for Luminal A")
plt.xlabel("Days")
plt.ylim(0, 1)
plt.ylabel("% of survival")
plt.show()

In [ ]:
expressions_luminal_A = luminal_A_df.iloc[:, 3:-1]

In [ ]:
df_life_line_LuminalA = luminal_A_df[
    list(expressions_luminal_A.columns) + ["event", "Overall Survival (Months)"]
]

In [ ]:
p_values_Cox_regression(df_life_line_LuminalA,event_col="event", duration_col="Overall Survival (Months)")


<font size="4">Luminal B, TNBC, Her-Enriched2 betas and p_values</font>

In [ ]:
diff_subtypes_df = comparation_df[comparation_df["Tumor-Cancer"] != "Luminal A"]
status = diff_subtypes_df["Overall Survival Status"].astype(str).str.strip()
diff_subtypes_df["event"] = status.str.contains("DECEASED", na=False) 
diff_subtypes_df = diff_subtypes_df.dropna(subset=["Overall Survival (Months)"])
X_diff_subtypes = diff_subtypes_df.iloc[:, 3:-1]
Y_surv_subtypes = Surv.from_dataframe(
    event="event",
    time="Overall Survival (Months)",
    data=diff_subtypes_df
)
